In [ ]:
# --- Librerías Estándar ---
import json
import time
import warnings
from collections import Counter, defaultdict
from pathlib import Path

# --- Librerías de Terceros (General) ---
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import cm
from tqdm.auto import tqdm

# --- Machine Learning y Datos ---
import open_clip
import torch
import torch.nn as nn
import torchvision.transforms.v2 as T
from datasets import load_dataset
from safetensors.torch import load_file
from sklearn.metrics import (
    confusion_matrix, 
    f1_score, 
    precision_score, 
    recall_score
)

from scipy.ndimage import gaussian_filter1d
# from openTSNE import TSNE

from sklearn.neighbors import KNeighborsClassifier
from torch.utils.data import DataLoader
from transformers import AutoModelForImageClassification

# --- Librerías Locales / Específicas ---
from planktonzilla.clip_model import *

# --- Configuración ---
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

In [ ]:
import sys
sys.path.append("/lustre/fswork/projects/rech/tec/uod68bo/am")

In [ ]:
#train_dataset = load_dataset("project-oceania/planktonzilla_only_plankton", split="train")
test_dataset = load_dataset("project-oceania/planktonzilla_only_plankton", split="test")

In [ ]:
train_dataset = load_dataset("project-oceania/planktonzilla_only_plankton", split="train")

train_labels = train_dataset["label"]
class_frequencies = dict(Counter(train_labels))

## Functions

In [ ]:
def clip_preds_and_features(
    model,
    dataset,
    tokenizer,
    preprocess,
    class_names,
    prompt_template=None,
    allowed_indices=None
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # 1. Enviar modelo a GPU primero
    model.to(device)
    
    # 2. Configurar Multi-GPU y definir helpers
    multi_gpu = False
    if torch.cuda.device_count() > 1:
        print(f"Usando {torch.cuda.device_count()} GPUs con DataParallel")
        model = nn.DataParallel(model)
        multi_gpu = True

    model.eval()

    # Helpers: Ahora model.module solo se accede si multi_gpu es True
    def encode_text(x): 
        return model.module.encode_text(x) if multi_gpu else model.encode_text(x)

    def encode_image(x): 
        return model.module.encode_image(x) if multi_gpu else model.encode_image(x)

    # 3. Procesar clases
    if prompt_template is not None:
        class_names = [prompt_template.format(label=c) for c in class_names]

    text_tok = tokenizer(class_names).to(device)
    with torch.no_grad():
        text_features = encode_text(text_tok)
        text_features /= text_features.norm(dim=-1, keepdim=True)

    # 4. Máscara de logits
    mask = None
    if allowed_indices is not None:
        mask = torch.full((len(class_names),), float('-inf')).to(device)
        mask[allowed_indices] = 0.0

    # 5. Dataloader
    def collate_fn(batch):
        images = torch.stack([preprocess(s["image"]) for s in batch])
        labels = torch.tensor([s["label"] for s in batch])
        return {"pixel_values": images, "labels": labels}

    dataloader = DataLoader(
        dataset, 
        batch_size=128, 
        shuffle=False, 
        num_workers=8, 
        collate_fn=collate_fn,
        pin_memory=True
    )

    all_labels, all_preds, all_features = [], [], []

    with torch.no_grad(), torch.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu"):
        for batch in tqdm(dataloader, desc="Inference"):
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"]

            image_features = encode_image(pixel_values)
            all_features.append(image_features.cpu())

            image_features /= image_features.norm(dim=-1, keepdim=True)
            
            logits = 100.0 * image_features @ text_features.T
            if mask is not None:
                logits = logits + mask 

            preds = logits.argmax(dim=1)

            all_labels.extend(labels.tolist())
            all_preds.extend(preds.cpu().tolist())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_features = torch.cat(all_features, dim=0).numpy()

    
    return all_labels, all_preds, all_features

In [ ]:
def clf_preds_and_features(model, dataset, transform):
    """
    Realiza inferencia en un dataset utilizando Multi-GPU si está disponible.
    Retorna etiquetas reales, predicciones y vectores de características (features).
    """
    
    def collate_fn(batch):
        pixel_values = torch.stack([x["pixel_values"] for x in batch])
        labels = torch.tensor([x["label"] for x in batch], dtype=torch.long)
        return {"pixel_values": pixel_values, "labels": labels}

    def preprocess(batch):
        # Aplicamos la transformación a la lista de imágenes "al vuelo"
        batch["pixel_values"] = [transform(img) for img in batch["image"]]
        return batch

    # Configurar el dataset para usar la función de preprocesamiento
    dataset.set_transform(preprocess)

    # Configuración del DataLoader
    dataloader = DataLoader(
        dataset, 
        batch_size=128, 
        shuffle=False, 
        collate_fn=collate_fn,
        num_workers=8,  
        pin_memory=True 
    )

    # Detectar y configurar el dispositivo
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Manejo de Multi-GPU con DataParallel
    if torch.cuda.device_count() > 1:
        print(f"🚀 ¡Activando Turbo! Usando {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
    
    model.to(device)
    model.eval()

    all_preds = []
    all_labels = []
    all_features = [] # Lista para acumular los tensores de características

    print(f"🧠 Iniciando inferencia en {device}...")

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Inferencia", unit="batch"):
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            # El modelo devuelve ImageClassifierOutput
            outputs = model(pixel_values)
            
            # 1. Obtener Logits y Predicciones
            logits = outputs.logits
            preds = logits.softmax(dim=1).argmax(dim=1)

            # 2. Obtener Features (vienen en hidden_states como una tupla)
            # Extraemos el primer elemento de la tupla: [Batch_Size, Num_Features]
            features = outputs.hidden_states[0]

            # Movemos a CPU y guardamos
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            all_features.append(features.cpu())

    # Concatenamos todos los batches de features en un único array de NumPy
    # Esto resulta en una matriz de forma (Total_Imagenes, Num_Features)
    final_features = torch.cat(all_features, dim=0).numpy()

    return all_labels, all_preds, final_features


In [ ]:
def evaluate_taxonomic_metrics(y_true, y_pred, class_names):
    """
    Retorna precision / recall / f1 macro por nivel taxonómico.
    Maneja clases con taxonomías incompletas evaluando la ruta disponible.

    Además calcula 'outside_parent_rate': el porcentaje de errores en un 
    nivel que son consecuencia de un error en el nivel taxonómico superior.
    """

    TAXONOMIC_LEVELS = [
        "kingdom", "phylum", "class", "order", 
        "family", "genus", "species"
    ]
    
    results = {}

    for level_idx, level_name in enumerate(TAXONOMIC_LEVELS):
        y_true_bin = []
        y_pred_bin = []

        # n_incorrect = 0
        # n_outside_parent = 0

        for yt, yp in zip(y_true, y_pred):
            true_tokens = class_names[yt].split()
            pred_tokens = class_names[yp].split()

            # Tomamos la taxonomía concatenada HASTA el nivel actual.
            # Si los tokens son más cortos que el nivel, Python maneja 
            # el slice [:level_idx + 1] sin arrojar error, trayendo todo lo disponible.
            true_label = " ".join(true_tokens[:level_idx + 1])
            pred_label = " ".join(pred_tokens[:level_idx + 1])

            y_true_bin.append(true_label)
            y_pred_bin.append(pred_label)

        #     # -------- MÉTRICA: outside_parent_rate --------
        #     if pred_label != true_label:
        #         n_incorrect += 1

        #         # Comparamos el "padre" (el string concatenado hasta el nivel anterior)
        #         if level_idx > 0:
        #             true_parent = " ".join(true_tokens[:level_idx])
        #             pred_parent = " ".join(pred_tokens[:level_idx])
                    
        #             if pred_parent != true_parent:
        #                 n_outside_parent += 1

        # # Cálculo de la tasa de error por padre incorrecto
        # outside_parent_rate = (
        #     n_outside_parent / n_incorrect if n_incorrect > 0 else 0.0
        # )

        results[level_name] = {
            "precision": precision_score(
                y_true_bin, y_pred_bin, zero_division=0, average='macro'
            ),
            "recall": recall_score(
                y_true_bin, y_pred_bin, zero_division=0, average='macro'
            ),
            "f1": f1_score(
                y_true_bin, y_pred_bin, zero_division=0, average='macro'
            ),
            # "n_incorrect": n_incorrect,
            # "n_outside_parent": n_outside_parent,
            # "outside_parent_rate": outside_parent_rate,
        }

    return results

In [ ]:
def apply_simpleshot(train_feats, test_feats, test_labels, k=1, seed=42):
    rng = np.random.default_rng(seed)
    
    mean_train = np.mean(train_feats, axis=0)

    def preprocess(x):
        x = x - mean_train
        norm = np.linalg.norm(x, axis=1, keepdims=True)
        return x / (norm + 1e-10)

    train_feats_proc = preprocess(train_feats)
    test_feats_proc = preprocess(test_feats)

    unique_classes = np.unique(test_labels)
    prototypes, proto_labels = [], []
    mask_prototypes = np.zeros(len(test_labels), dtype=bool)
    
    for cls in unique_classes:
        cls_indices = np.where(test_labels == cls)[0]
        if len(cls_indices) > 0:
            n_take = min(len(cls_indices), k)
            selected = rng.choice(cls_indices, size=n_take, replace=False)
            mask_prototypes[selected] = True
            
            proto = np.mean(test_feats_proc[selected], axis=0)
            prototypes.append(proto)
            proto_labels.append(cls)

    X_proto, y_proto = np.array(prototypes), np.array(proto_labels)
    X_query, y_query = test_feats_proc[~mask_prototypes], test_labels[~mask_prototypes]

    knn = KNeighborsClassifier(n_neighbors=1, metric='euclidean', n_jobs=32)
    knn.fit(X_proto, y_proto)
    y_pred = knn.predict(X_query)

    return y_query, y_pred

In [ ]:
def benchmark_simpleshot(train_paths, k=1, seeds=[0, 7, 25, 42, 100]):
    """
    Realiza el benchmark de SimpleShot para múltiples experimentos y promedia resultados.
    
    Args:
        train_paths (dict): Diccionario {nombre_exp: path_a_embeddings_train}.
        k (int): Número de ejemplos por clase para formar los prototipos.
        seeds (list): Lista de semillas para promediar el sampleo.
    """
    taxa_levels = ['kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']
    
    # Encabezado de la tabla
    header = f"{'Experimento':<40} | " + " | ".join([f"{t.capitalize():<6}" for t in taxa_levels])
    print(f"\nResultados SimpleShot (k={k}, runs={len(seeds)})")
    print(header)
    print("-" * len(header))

    for exp_name, train_path in train_paths.items():
        # Construir path de test basado en el nombre del experimento
        test_path = f"/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/tutorials/{exp_name}_test_results.npz"
        
        # Carga de datos
        try:
            train_feats = np.load(train_path)
            test_data = np.load(test_path)
        except FileNotFoundError:
            print(f"Error: No se encontraron archivos para {exp_name}")
            continue

        y_test_true = test_data['y_true']
        test_feats = test_data['features']

        accumulated_f1 = {t: [] for t in taxa_levels}
        # Ejecución de los runs con distintas semillas
        for s in seeds:
            y_true_ss, y_pred_ss = apply_simpleshot(
                train_feats, test_feats, y_test_true, k=k, seed=s
            )
            
            # 1. Métricas Taxonómicas (promediando por run)
            r = evaluate_taxonomic_metrics(y_true_ss, y_pred_ss, test_dataset.features["label"].names)
            for t in taxa_levels:
                accumulated_f1[t].append(r.get(t, {}).get('f1', 0.0))
            
        # Cálculo de promedios para la fila
        row_values = []
        for t in taxa_levels:
            avg_taxa_f1 = np.mean(accumulated_f1[t])
            row_values.append(f"{avg_taxa_f1:<6.3f}")
        
        
        # Imprimir fila del experimento
        print(f"{exp_name[:40]:<40} | " + " | ".join(row_values))

In [ ]:
def hierarchical_tsne_plot(
    dataset,
    features,
    hierarchy_dict,
    levels_order=("kingdom", "phylum", "class", "order", "family", "genus", "species"),
    perplexity=5,
    n_jobs=96,
    figsize=(20, 10)
):

    def parse_taxonomy(label):
        parts = label.split()
        parts = parts + [None] * (7 - len(parts))
        return parts[:7]

    taxonomy = pd.DataFrame(
        [parse_taxonomy(l) for l in dataset.features["label"].names],
        columns=levels_order
    )

    y_true = np.array(dataset["label"])

    df = pd.DataFrame({"y_true": y_true})
    df = df.join(taxonomy, on="y_true")

    selected_levels = [lvl for lvl in levels_order if lvl in hierarchy_dict]

    root_level = selected_levels[0]
    root_target = hierarchy_dict[root_level]

    mask_root = df[root_level] == root_target

    X_root = features[mask_root]
    df_root = df[mask_root].reset_index(drop=True)

    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        n_jobs=n_jobs,
        initialization="pca"
    )

    X_tsne = tsne.fit(X_root)

    x_min, x_max = X_tsne[:, 0].min() - 5, X_tsne[:, 0].max() + 5
    y_min, y_max = X_tsne[:, 1].min() - 5, X_tsne[:, 1].max() + 5

    n_plots = len(selected_levels)
    n_cols = min(3, n_plots)
    n_rows = int(np.ceil(n_plots / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)

    if n_plots == 1:
        axes = np.array([axes])
    else:
        axes = np.array(axes).reshape(-1)

    cmap = plt.get_cmap("tab20")

    current_mask = np.ones(len(df_root), dtype=bool)

    max_legend_rows = 1

    for i, level in enumerate(selected_levels):
        ax = axes[i]

        target_value = hierarchy_dict[level]
        current_mask = current_mask & (df_root[level] == target_value)

        df_plot = df_root[current_mask]
        X_plot = X_tsne[current_mask]

        level_idx = levels_order.index(level)
        if level_idx + 1 < len(levels_order):
            child_level = levels_order[level_idx + 1]
        else:
            child_level = level

        categories = df_plot[child_level].dropna().unique()

        for j, cat in enumerate(categories):
            idx = df_plot[child_level] == cat
            ax.scatter(
                X_plot[idx, 0],
                X_plot[idx, 1],
                s=5,
                alpha=0.7,
                color=cmap(j % 20),
                label=cat
            )

        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)

        ax.set_xticks([])
        ax.set_yticks([])

        ax.set_title(f"{child_level.capitalize()} of {target_value.capitalize()}", fontsize=16)

        # --- LEYENDA ABAJO (FIXED) ---
        n_cats = len(categories)
        if n_cats > 0:
            n_cols_legend = min(3, n_cats)

            # calcular filas de leyenda
            n_rows_legend = int(np.ceil(n_cats / n_cols_legend))
            max_legend_rows = max(max_legend_rows, n_rows_legend)

            ax.legend(
                markerscale=2,
                loc="upper center",
                bbox_to_anchor=(0, -0.1, 1, 0),  
                ncol=min(3, n_cats),
                fontsize="small",
                frameon=True
            )

    for j in range(len(selected_levels), len(axes)):
        axes[j].axis("off")

    fig.subplots_adjust(
        hspace=0.15,  
        wspace=0.1,    
        bottom=0.1 + 0.05 * max_legend_rows
    )

    plt.show()

In [ ]:
#import open_clip
#import torch

#model, _, _ = open_clip.create_model_and_transforms('')

#checkpoint_path = "/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/logs/2026_02_27-06_51_03-model_hf-hub:imageomics-bioclip-lr_0.0001-b_512-j_4-p_amp/checkpoints/epoch_95.pt"
#checkpoint_save = "/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/logs/2026_02_27-06_51_03-model_hf-hub:imageomics-bioclip-lr_0.0001-b_512-j_4-p_amp/checkpoints/epoch_95_fixed.pt"
#device = "cpu"

#checkpoint = torch.load(checkpoint_path, map_location=device)

#state_dict = checkpoint['state_dict']

# SOLUCIÓN DE PROBLEMAS COMUNES (DataParallel):
# Si entrenaste con múltiples GPUs, las keys pueden tener el prefijo 'module.'
#new_state_dict = {}
#for key, value in state_dict.items():
#    if key.startswith('module.'):
#        new_state_dict[key[7:]] = value
#    else:
#        new_state_dict[key] = value

# Cargar los pesos al modelo
#msg = model.load_state_dict(new_state_dict, strict=True)
#print(f"Pesos cargados. Reporte: {msg}")

#checkpoint['state_dict'] = model.cpu().state_dict()
#torch.save(checkpoint, checkpoint_save)

## CLIP

In [ ]:
modelos = {
           "ViT-B-16 OpenAI-Planktonzilla": 'hf-hub:project-oceania/CLIP-ViT-B-16.openai-pt.planktonzilla-pt',
           "ViT-B-16 BioCLIP-Planktonzilla": 'hf-hub:project-oceania/CLIP-ViT-B-16.bioclip-pt.planktonzilla-pt',
           #"ViT-B-16 BioCLIP": 'hf-hub:imageomics/bioclip',
           #"ViT-L-14 Laion2b-Planktonzilla": 'hf-hub:project-oceania/CLIP-ViT-L-14.laion2b-pt.planktonzilla-pt',
           #"ViT-L-14 BioCLIP2-Planktonzilla": 'hf-hub:project-oceania/CLIP-ViT-L-14.bioclip2-pt.planktonzilla-pt',
           #"ViT-L-14 BioCLIP2": 'hf-hub:imageomics/bioclip-2',
          }

In [ ]:
class_names = test_dataset.features["label"].names

for exp_name, model_name in modelos.items():

    
    model, _, preprocess = open_clip.create_model_and_transforms(model_name)
    tokenizer = open_clip.get_tokenizer(model_name)

    _, _, all_features = clip_preds_and_features(
                                            model,
                                            train_dataset,
                                            tokenizer,
                                            preprocess,
                                            class_names,
                                            prompt_template=None
                                        )

    np.save(f"/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/tutorials/{exp_name}_train_features.npy", all_features)

In [ ]:
test_labels = np.array(test_dataset["label"])
test_unique_indices = np.unique(test_labels).tolist() 
class_names = test_dataset.features["label"].names

my_prompt = "a photo of a {label}"

for exp_name, model_name in modelos.items():
    model, _, preprocess = open_clip.create_model_and_transforms(model_name)
    tokenizer = open_clip.get_tokenizer(model_name)

    # Pasamos los índices únicos como máscara
    targets, preds, all_features = clip_preds_and_features(
        model,
        test_dataset, 
        tokenizer,
        preprocess,
        class_names, 
        prompt_template=my_prompt,
        allowed_indices=test_unique_indices 
    )

    output_path = f"/lustre/fswork/projects/rech/tec/uod68bo/am/open_clip/tutorials/{exp_name}_with_promp_test_results.npz"
    np.savez(output_path, features=all_features, y_true=targets, y_preds=preds)

In [ ]:
experimentos = {"ViT-B-16 OpenAI-Planktonzilla": 'ViT-B-16 OpenAI-Planktonzilla_with_promp_test_results.npz',
                           "ViT-B-16 BioCLIP-Planktonzilla": 'ViT-B-16 BioCLIP-Planktonzilla_with_promp_test_results.npz',
                           "ViT-B-16 BioCLIP": 'ViT-B-16 BioCLIP_with_promp_test_results.npz',
                           "ViT-L-14 Laion2b-Planktonzilla": 'ViT-L-14 Laion2b-Planktonzilla_with_promp_test_results.npz',
                           "ViT-L-14 BioCLIP2-Planktonzilla": 'ViT-L-14 BioCLIP2-Planktonzilla_with_promp_test_results.npz',
                           "ViT-L-14 BioCLIP2": 'ViT-L-14 BioCLIP2_with_promp_test_results.npz',
                          }

In [ ]:
taxa_levels = ['kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']

# Encabezado de la tabla
header = f"{'Experimento':<40} | " + " | ".join([f"{t.capitalize():<8}" for t in taxa_levels]) + " | Original F1"
separator = "-" * len(header)

print(header)
print(separator)

for exp_name, results_path in experimentos.items():
    # Cargar datos
    data = np.load(results_path) # Usamos la variable results_path del loop
    y_true = data["y_true"]
    y_pred = data["y_preds"]
    
    f1 = f1_score(y_true, y_pred,
                average="macro", zero_division=0)

    # Calcular métricas taxonómicas
    r = evaluate_taxonomic_metrics(y_true, y_pred, test_dataset.features["label"].names)
    
    # Extraer f1 de cada nivel 
    row_values = []
    for t in taxa_levels:
        val = r.get(t, {}).get('f1', 0.0)
        row_values.append(f"{val:<8.3f}")
    
    
    # Imprimir fila formateada
    print(f"{exp_name[:40]:<40} | " + " | ".join(row_values) + f" | {f1:.3f}")

In [ ]:
train_paths = {
               "ViT-B-16 BioCLIP": 'ViT-B-16 BioCLIP_train_features.npy',
               "ViT-L-14 BioCLIP2": 'ViT-L-14 BioCLIP2_train_features.npy',
              }

In [ ]:
benchmark_simpleshot(train_paths, k=1)


In [ ]:
benchmark_simpleshot(train_paths, k=5)

In [ ]:
benchmark_simpleshot(train_paths, k=10)